In [ ]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

In [ ]:
df_2024_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/Preços semestrais - AUTOMOTIVOS_2024.01.csv", sep=';')

In [ ]:
df_2024_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/Preços semestrais - AUTOMOTIVOS_2024.02.csv", sep=';')

In [ ]:
df_2024 = pd.concat((df_2024_01,df_2024_02))

In [ ]:
df_2024.info()

<class 'pandas.DataFrame'>
Index: 898536 entries, 0 to 421381
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     898536 non-null  str    
 1   Estado - Sigla     898536 non-null  str    
 2   Municipio          898536 non-null  str    
 3   Revenda            898536 non-null  str    
 4   CNPJ da Revenda    898536 non-null  str    
 5   Nome da Rua        898536 non-null  str    
 6   Numero Rua         898371 non-null  str    
 7   Complemento        205528 non-null  str    
 8   Bairro             897095 non-null  str    
 9   Cep                898536 non-null  str    
 10  Produto            898536 non-null  str    
 11  Data da Coleta     898536 non-null  str    
 12  Valor de Venda     898536 non-null  str    
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  898536 non-null  str    
 15  Bandeira           898536 non-null  str    
dtypes: float64(1), str

In [ ]:
df_2024_pe = df_2024[df_2024["Estado - Sigla"] == "PE"]

In [ ]:
df_2024_pe.head()

In [ ]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Staging no BigQuery
table_id = os.getenv("BRONZE_2024")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [ ]:
# Criação e carga da tabela de STG

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela de staging seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2024_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print(f"✅ Tabela Stg.{table_id} criada e dados carregados com sucesso")